# BIFROST: the source through the chopper train

The ESS source BIFROST sees, as neutrons per second into its guide's entrance over
(emission time, wavelength): McStas's `ESS_butterfly` moderator for beamline W4, computed
rather than sampled by `ess_butterfly.py`. chopcal transmits the source rectangle through BIFROST's six choppers
exactly, as convex polygons -- the outline drawn over the source -- and weights it by how
likely each neutron is to get through while a disc's edge is still sweeping the beam.

On the right is what reaches the sample: by arrival time, and by wavelength (energy on the
top axis). The dashed curves count the whole transmitted region as open; the solid ones
weight it by the transmission. In green, on the wavelength pane's right-hand axis, is the
spread of wavelength $\sigma_\lambda$ among the neutrons that reach the sample together,
against their mean wavelength: how well the arrival time pins the wavelength down. It is
chopcal's exact `arrival_moments`, one point per arrival-time bin.

Below is a time-distance diagram. Each band is what got through every chopper before one,
from the source to that chopper -- each polygon's times at a distance $L$ are
$t + L/v$ over its vertices -- drawn translucent so that overlaps show. The part of a band
the next one does not cover is what that chopper stops; the last one, in red, reaches the
sample. Each chopper is a line broken where it is open, widened by the beam's width. The panel on
the right zooms in on the choppers nearest the source, where most of the source is stopped.

The controls set where chopcal puts the band (lowest energy, pulse-shaping time), the
source rectangle (latest emission, wavelength range), the grid the source is binned on,
the proton pulse's duration, the beam power at the full pulse (the proton current is
held, so a shorter pulse is less power), a factor on the moderator's decay times
(the decay is fitted over the first few pulse lengths, and extrapolated beyond), a pulse
delay that moves every disc together, as `BeamPosDly-S` does on the real
choppers, and how fast the pulse-shaping disks may turn: the designed ones are rated to
210 Hz, the substitutes installed now to 126 Hz. chopcal turns them at the fastest
harmonic of 14 Hz up to that limit that the burst allows, and the speed it picks is shown
under the control.

*guide transmission*, on by default, adds what reaches the sample through the real
guide to the arrival-time plot, in red on its own axis, because it is an order of
magnitude below the perfect-guide curves. It also makes the wavelength plot show
intensity through the guide, and the source on the left weighted by the guide. The
weight is the fraction of what enters the guide that reaches a 10 x 30 mm sample window.
It comes from a McStas scan of niess's BIFROST with the choppers removed
(`guide_transmission.csv`): almost nothing below 1 Å, rising to about 8 % beyond 8 Å.

Under them is each chopper's delay. It follows `chopcal.bifrost` whenever the lowest
energy, the shaping time or the PSC limit changes, unless its lock is pressed. Any delay
can be moved by hand, and stays where it is put until one of those three changes. Beside
each is chopcal's delay for the current inputs, for comparison. Speeds are always
chopcal's, and the pulse delay still moves every disc on top of these.

Needs `chopcal>=0.8`, `ipympl` and `ipywidgets`.

In [ ]:
%matplotlib widget

from polygonfigure import bifrost_transmitted_phasespace_figure

In [ ]:
bifrost_transmitted_phasespace_figure(ei_min=4.5, t_psc=0.2)

## The guide's transmission

The fraction of what enters BIFROST's guide that reaches a 10 x 30 mm sample window,
against wavelength: the table the *guide transmission* toggle weights the source by. From
a McStas scan of niess's BIFROST with the choppers removed, 10⁶ rays per point.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from polygonfigure import GUIDE_TRANSMISSION

wavelength, fraction, error = np.loadtxt(GUIDE_TRANSMISSION, delimiter=',', skiprows=1).T
fig, ax = plt.subplots(figsize=(7, 3.5), layout='constrained')
ax.errorbar(wavelength, 100 * fraction, 100 * error, fmt='.-', markersize=3)
ax.set_xlabel('wavelength [Å]')
ax.set_ylabel('guide transmission [%]')
ax.set_ylim(bottom=0)
plt.show()

## Without widgets

`transmission` computes one setting and `draw` puts it on a figure, for a script or a
saved picture.

In [ ]:
from polygonfigure import draw, make_axes, transmission

result = transmission(energy_min=4.5, shaping_time=2e-4, pulse_delay=0.0)
fig, axes = make_axes()
draw(result, axes)
print(f'{result.transmitted:.3g} n/s, {result.transmitted / result.emitted:.3%} of the '
      'source rectangle, reach the sample')